In [7]:
import pandas as pd

alvr_og = pd.read_excel("../data_storage/raw_data/voter_data/2024_votereg_data_AlSoS.xlsx", sheet_name="October", header=None, engine="openpyxl")
rdh_og = pd.read_csv("../data_storage/raw_data/voter_data/AL_l2_2024_gen_stats_2020county.csv")
demographics_og = pd.read_csv("../data_storage/clean_data/eligible_population_2020_general/eligible_population_2020_general.csv")

alvr = alvr_og.copy()
rdh = rdh_og.copy()
demographics = demographics_og.copy()

#RDH uses different column names so this renames and standardizes data
rdh = rdh.rename(columns={"countyname": "county_name", "countyfips": "fips_code"})
rdh["county_name"] = rdh["county_name"].str.strip().str.title()

#ALVR has a broken two-row header so manually set clean column names
alvr.columns = [
    'county_name', 'Total_AI',
    'Active_Asian', 'Active_AmInd', 'Active_Black', 'Active_Fed',
    'Active_Hispanic', 'Active_Korean', 'Active_White', 'Active_Other',
    'Active_NotId', 'Active_Total',
    'Inactive_Asian', 'Inactive_AmInd', 'Inactive_Black', 'Inactive_Fed',
    'Inactive_Hispanic', 'Inactive_Korean', 'Inactive_White', 'Inactive_Other',
    'Inactive_NotId', 'Inactive_Total'
]

#Drops rows 0 and 1 which contained the broken header labels then resets index
alvr = alvr.iloc[2:].reset_index(drop=True)
alvr["county_name"] = alvr["county_name"].astype(str).str.strip().str.title()

#Fill NaN in ALVR columns with 0 to handle missing values like Wilcox Inactive Hispanic
alvr_numeric_columns = [
    'Active_Asian', 'Active_AmInd', 'Active_Black', 'Active_Fed', 'Active_Hispanic',
    'Active_Korean', 'Active_White', 'Active_Other', 'Active_NotId', 'Active_Total',
    'Inactive_Asian', 'Inactive_AmInd', 'Inactive_Black', 'Inactive_Fed', 'Inactive_Hispanic',
    'Inactive_Korean', 'Inactive_White', 'Inactive_Other', 'Inactive_NotId', 'Inactive_Total'
]
alvr[alvr_numeric_columns] = alvr[alvr_numeric_columns].fillna(0)

#Filters the 17 counties for Alabama BB
BLACK_BELT_COUNTIES = [
    "Sumter", "Choctaw", "Greene", "Hale", "Marengo", "Perry", "Dallas",
    "Wilcox", "Lowndes", "Butler", "Crenshaw", "Montgomery", "Pike",
    "Bullock", "Macon", "Barbour", "Russell"
]

alvr_bb = alvr[alvr["county_name"].isin(BLACK_BELT_COUNTIES)].copy()
rdh_bb = rdh[rdh["county_name"].isin(BLACK_BELT_COUNTIES)].copy()

#Inner join to merge ALVR with RDH for FIPS code and demographics for eligible population
registration = alvr_bb.merge(rdh_bb[["county_name", "fips_code"]], on="county_name")
registration = registration.merge(demographics[["county_name", "eligible_black", "eligible_white", "eligible_latinx"]], on="county_name")

registration["total_registered_black"] = registration["Active_Black"].astype(float) + registration["Inactive_Black"].astype(float)
registration["total_registered_white"] = registration["Active_White"].astype(float) + registration["Inactive_White"].astype(float)
registration["total_registered_latinx"] = registration["Active_Hispanic"].astype(float) + registration["Inactive_Hispanic"].astype(float)

#This calculates active registration rates: active divided by eligible population
#This can be used to show what share of eligible voters are actively registered and ready to vote
registration["reg_active_rate_black"] = (registration["Active_Black"].astype(float) / registration["eligible_black"]).round(4)
registration["reg_active_rate_white"] = (registration["Active_White"].astype(float) / registration["eligible_white"]).round(4)
registration["reg_active_rate_latinx"] = (registration["Active_Hispanic"].astype(float) / registration["eligible_latinx"]).round(4)

#This calculates active share by race (active divided by total registered)
#Uses ALVR only to avoid CVAP undercount issues in rural majority-minority counties
#Lower value indicates higher purge risk
registration["active_share_black"] = (registration["Active_Black"].astype(float) / registration["total_registered_black"]).round(4)
registration["active_share_white"] = (registration["Active_White"].astype(float) / registration["total_registered_white"]).round(4)
registration["active_share_latinx"] = (registration["Active_Hispanic"].astype(float) / registration["total_registered_latinx"]).round(4)

#This calculates inactive rates to flag purge risk
#The overall inactive rate = total inactive divided by total registered
registration["inactive_rate"] = (registration["Inactive_Total"].astype(float) / (registration["Active_Total"].astype(float) + registration["Inactive_Total"].astype(float))).round(4)

#The inactive rate for each race
registration["inactive_rate_black"] = (registration["Inactive_Black"].astype(float) / registration["total_registered_black"]).round(4)
registration["inactive_rate_white"] = (registration["Inactive_White"].astype(float) / registration["total_registered_white"]).round(4)
registration["inactive_rate_latinx"] = (registration["Inactive_Hispanic"].astype(float) / registration["total_registered_latinx"]).round(4)

#Rename ALVR columns to follow naming convention: lowercase with underscores
registration = registration.rename(columns={
    #Active registered by race
    "Active_Black": "active_black",
    "Active_White": "active_white",
    "Active_Hispanic": "active_latinx",
    "Active_Total": "active_total",
    
    #Inactive registered by race 
    "Inactive_Black": "inactive_black",
    "Inactive_White": "inactive_white",
    "Inactive_Hispanic": "inactive_latinx",
    "Inactive_Total": "inactive_total"
})

final_columns = [
    #Geographic identifiers
    "county_name", "fips_code",

    #Overall totals across all races
    "active_total", "inactive_total", "inactive_rate",

    #Black voters
    "eligible_black", "active_black", "inactive_black", "total_registered_black",
    "active_share_black", "inactive_rate_black",

    #White voters
    "eligible_white", "active_white", "inactive_white", "total_registered_white",
    "active_share_white", "inactive_rate_white",

    #Latinx voters
    "eligible_latinx", "active_latinx", "inactive_latinx", "total_registered_latinx",
    "active_share_latinx", "inactive_rate_latinx"]

registration = registration[final_columns].sort_values("county_name").reset_index(drop=True)

#Convert missing values to null per standardization document
registration = registration.fillna("null")

#Displays final cleaned registration dataset
registration.to_csv("../data_storage/clean_data/voter_registration_2020_general/voter_registration_2020_general.csv", index=False)
print("Created: voter_registration_2020_general.csv")
registration

Created: voter_registration_2020_general.csv


/var/folders/_7/6hr00gjd7rgcmfz744n7rc2h0000gn/T/ipykernel_92659/2140701844.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  alvr[alvr_numeric_columns] = alvr[alvr_numeric_columns].fillna(0)


,county_name,fips_code,active_total,inactive_total,inactive_rate,eligible_black,active_black,inactive_black,total_registered_black,active_share_black,...,inactive_white,total_registered_white,active_share_white,inactive_rate_white,eligible_latinx,active_latinx,inactive_latinx,total_registered_latinx,active_share_latinx,inactive_rate_latinx
0,Barbour,1005,15697,1956,0.1108,9245,7171,963,8134.0,0.8816,...,922,9053.0,0.8982,0.1018,405,142,29,171.0,0.8304,0.1696
1,Bullock,1011,6376,803,0.1119,5525,4789,656,5445.0,0.8795,...,132,1607.0,0.9179,0.0821,250,63,7,70.0,0.9000,0.1000
2,Butler,1013,13236,1287,0.0886,6564,5797,700,6497.0,0.8923,...,557,7824.0,0.9288,0.0712,175,39,7,46.0,0.8478,0.1522
3,Choctaw,1023,9624,1140,0.1059,4259,4088,618,4706.0,0.8687,...,505,5978.0,0.9155,0.0845,55,18,2,20.0,0.9000,0.1000
4,Crenshaw,1041,10010,835,0.0770,2564,2414,247,2661.0,0.9072,...,551,7990.0,0.9310,0.0690,125,42,13,55.0,0.7636,0.2364
5,Dallas,1047,26569,2942,0.0997,19904,18687,2252,20939.0,0.8924,...,618,8175.0,0.9244,0.0756,80,53,13,66.0,0.8030,0.1970
6,Greene,1063,6077,495,0.0753,5085,4907,412,5319.0,0.9225,...,72,1172.0,0.9386,0.0614,40,17,4,21.0,0.8095,0.1905
7,Hale,1065,11345,814,0.0669,6664,6498,493,6991.0,0.9295,...,302,5002.0,0.9396,0.0604,10,29,6,35.0,0.8286,0.1714
8,Lowndes,1085,8811,852,0.0882,5654,6465,660,7125.0,0.9074,...,175,2450.0,0.9286,0.0714,30,19,8,27.0,0.7037,0.2963
9,Macon,1087,13040,3933,0.2317,12499,10381,3596,13977.0,0.7427,...,224,2613.0,0.9143,0.0857,60,60,21,81.0,0.7407,0.2593
